In [1]:
# importing some libraries 

import pandas as pd
import matplotlib.pyplot as plt 

# setting the plot dpi 
# plt.rcParams['figure.dpi']= 800

In [2]:
# setting up / configuring 
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "training_linear_models"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [3]:
# reading the csv file and printing the head 
df = pd.read_csv("C:/Users/andsa/Documents/ML/concrete_data.csv")  
df.head()


# Checking for correlation 
corr_matrix = df.corr()
corr_matrix["Strength"]


Cement                0.497832
Blast Furnace Slag    0.134829
Fly Ash              -0.105755
Water                -0.289633
Superplasticizer      0.366079
Coarse Aggregate     -0.164935
Fine Aggregate       -0.167241
Age                   0.328873
Strength              1.000000
Name: Strength, dtype: float64

In [4]:
X = df[['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']]
y = df[['Strength']]

In [5]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling the data - important for linear regression 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)





In [6]:
# Doing an ordinary least squares regression 
from sklearn.linear_model import LinearRegression
lin_reg_OLS = LinearRegression()
lin_reg_OLS.fit(X_train, y_train)


# Doing a stochastic gradient descent method -> mini-batch
from sklearn.linear_model import SGDRegressor
#lin_reg_SGD = SGDRegressor(max_iter=1300, eta0=0.01, learning_rate="constant")
#lin_reg_SGD = SGDRegressor(max_iter = 4000, learning_rate= "optimal", random_state = 42)
lin_reg_SGD = SGDRegressor(max_iter=5000, tol=1e-5, eta0=0.01, learning_rate='constant', random_state=42)

lin_reg_SGD.fit(X_train_scaled, y_train)

c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


SGDRegressor(learning_rate='constant', max_iter=5000, random_state=42,
             tol=1e-05)

In [ ]:
# Predicting strength values on the pred set 

y_pred_OLS = lin_reg_OLS.predict(X_test)

y_pred_SGD = lin_reg_SGD.predict(X_test_scaled)


print("OLS Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_OLS), 4)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_OLS), 2)}\n")


print("Mini-batch SGD Regression:")
print(f"R² Score: {np.round(r2_score(y_test, y_pred_SGD), 2)}")
print(f"MSE: {np.round(mean_squared_error(y_test, y_pred_SGD), 2)}")



OLS Regression:
R² Score: 0.63
MSE: 95.9709

Mini-batch SGD Regression:
R² Score: 0.6303
MSE: 95.2739


In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import SGDRegressor

# Define hyperparameter grid with some values
param_grid = {
    "eta0": [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],  # More values
    "max_iter": [500, 1000, 1500, 2000, 3000, 5000],  # Increased range
    "learning_rate": ["constant", "optimal", "invscaling", "adaptive"]
}

# Create SGDRegressor
sgd = SGDRegressor(random_state=42)

# Perform Grid Search
grid_search = GridSearchCV(sgd, param_grid, cv=5, scoring="r2", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# Get the best model
best_sgd = grid_search.best_estimator_

# Predict on the test set
y_pred = best_sgd.predict(X_test_scaled)

# Compute RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mse = mean_squared_error(y_test, y_pred)

# Print results
print("Best Parameters:", grid_search.best_params_)
print("Best R² Score (CV):", grid_search.best_score_)
print("Test RMSE:", rmse)
print("Test MSE:", mse)




Best Parameters: {'eta0': 0.1, 'learning_rate': 'adaptive', 'max_iter': 500}
Best R² Score (CV): 0.5945867371010949
Test RMSE: 9.791266768929757
Test MSE: 95.86890494034817


c:\Users\andsa\anaconda3\lib\site-packages\sklearn\utils\validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


# cross validation chap 2
over or underfitting

learning curves for poly?


# poly fit 

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

polynomial_regression = Pipeline([
        ("poly_features", PolynomialFeatures(degree=3, include_bias=False)),
        ("lin_reg", LinearRegression()),
    ])

plot_learning_curves(polynomial_regression, X, y)
plt.axis([0, 1030, 0, 100])           # not shown
save_fig("learning_curves_plot")  # not shown
plt.show()     





# from chat 
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Step 1: Create polynomial features (degree=2 for quadratic, you can adjust the degree)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_train_scaled)

# Step 2: Train a Linear Regression model on the transformed features
poly_reg = LinearRegression()
poly_reg.fit(X_poly, y_train)

# Step 3: Make predictions using the polynomial model
X_test_poly = poly.transform(X_test_scaled)  # Transform the test set as well
y_pred_poly = poly_reg.predict(X_test_poly)

# Step 4: Calculate RMSE and R² score
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = poly_reg.score(X_test_poly, y_test)

# Print the results
print(f"Polynomial Regression R² Score: {r2_poly}")
print(f"Polynomial Regression RMSE: {rmse_poly}")
